# CareLoop Clinics — No-Show Guard
## Part 4: GenAI Follow-up Review & Recommendation

### Task 4.1 — One validated Gemini call

In [ ]:
!pip install -q google-genai pydantic

In [ ]:
import json
import time
import pandas as pd
from google import genai
from google.genai import types
from pydantic import BaseModel, Field, ValidationError

MODEL_NAME = "gemini-2.5-flash"  # Update to your Session 22.1 Flash model if different
ALLOWED_ROOT_CAUSES = [
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other",
]

SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {"type": "string"},
        "confidence": {"type": "number"},
        "summary": {"type": "string"},
        "recommended_action": {"type": "string"},
    },
    "required": ["root_cause", "confidence", "summary", "recommended_action"],
}

SYSTEM_TEXT = (
    "You are a clinical operations analyst at CareLoop Clinics. "
    "You classify call-centre follow-up notes for missed appointments into structured JSON."
)

def build_craft_prompt(note_text):
    return (
        "Context: CareLoop Clinics call-centre staff write a short note after calling a patient who missed an appointment.\n"
        "Role: Act as an outpatient clinic operations analyst.\n"
        f"Action: Classify the note below into exactly one root_cause from this allowed list: {', '.join(ALLOWED_ROOT_CAUSES)}. "
        "If the note is ambiguous or does not clearly fit one of the first five categories, answer 'other' rather than guessing. "
        "Provide a confidence score from 0 to 1, a summary of at most 25 words, and a concrete recommended_action.\n"
        "Tone: Concise, neutral, and operational.\n"
        "Format: Return valid JSON matching the schema.\n\n"
        f"Note: {note_text}"
    )

API_KEY = input("Paste your Gemini API key: ")
client = genai.Client(api_key=API_KEY)
api_call_count = 0

def analyse_note_raw(note_text, prompt_text=None):
    global api_call_count
    if prompt_text is None:
        prompt_text = build_craft_prompt(note_text)
    api_call_count += 1
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt_text,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1000,
            response_mime_type="application/json",
            response_schema=SCHEMA,
            thinking_config=types.ThinkingConfig(thinking_level="minimal"),
        ),
    )
    return response.text

def analyse_note(note_text):
    raw = analyse_note_raw(note_text)
    return json.loads(raw)

notes_df = pd.read_csv("data/followup_notes.csv")
n001_result = analyse_note(notes_df.loc[0, "note_text"])
print("N001 result:", n001_result)

### Task 4.2 — Validate and process the first 10 notes

In [ ]:
class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

records = []
first_10 = notes_df.head(10)

for idx, row in first_10.iterrows():
    note_id = row["note_id"]
    appt_id = row["appointment_id"]
    note_text = row["note_text"]

    if pd.isna(note_text) or str(note_text).strip() == "":
        records.append({
            "note_id": note_id,
            "appointment_id": appt_id,
            "root_cause": "other",
            "confidence": 0.0,
            "summary": "No note recorded",
            "recommended_action": "Re-call patient",
            "validation_status": "skipped_blank",
            "raw_response": "",
        })
        continue

    time.sleep(13)  # respect free-tier rate limit
    raw_text = ""
    try:
        raw_text = analyse_note_raw(note_text)
        parsed = json.loads(raw_text)
        validated = FollowupInsight(**parsed)
        is_allowed = validated.root_cause in ALLOWED_ROOT_CAUSES
        status = "valid" if is_allowed else "invalid"
        records.append({
            "note_id": note_id,
            "appointment_id": appt_id,
            "root_cause": parsed.get("root_cause", ""),
            "confidence": parsed.get("confidence", ""),
            "summary": parsed.get("summary", ""),
            "recommended_action": parsed.get("recommended_action", ""),
            "validation_status": status,
            "raw_response": raw_text,
        })
    except Exception as e:
        parsed = {}
        try:
            parsed = json.loads(raw_text) if raw_text else {}
        except Exception:
            parsed = {}
        records.append({
            "note_id": note_id,
            "appointment_id": appt_id,
            "root_cause": parsed.get("root_cause", ""),
            "confidence": parsed.get("confidence", ""),
            "summary": parsed.get("summary", ""),
            "recommended_action": parsed.get("recommended_action", ""),
            "validation_status": "invalid",
            "raw_response": raw_text,
        })

notes_analysed = pd.DataFrame(records)
notes_analysed.to_csv("notes_analysed.csv", index=False)
print(notes_analysed[["note_id", "appointment_id", "root_cause", "confidence", "validation_status"]])

### Task 4.3 — Break it on purpose

In [ ]:
# (a) Deliberately bad Pydantic inputs
bad_conf = {
    "root_cause": "other",
    "confidence": 1.7,
    "summary": "Patient had a schedule conflict.",
    "recommended_action": "Offer evening slot"
}
bad_missing = {
    "root_cause": "other",
    "confidence": 0.8,
    "summary": "Patient had a schedule conflict."
}

for label, payload in [("confidence=1.7", bad_conf), ("missing recommended_action", bad_missing)]:
    try:
        FollowupInsight(**payload)
    except ValidationError as err:
        print(f"ValidationError for {label}:\n{err}\n")

# (b) Compare V1 vs V2 on N007 (Hinglish) and N009 (ambiguous)
v1_v2_rows = []
for target_id in ["N007", "N009"]:
    txt = notes_df.loc[notes_df["note_id"] == target_id, "note_text"].values[0]

    time.sleep(13)
    raw_v1 = analyse_note_raw(txt, prompt_text=f"Classify this note: {txt}")
    d_v1 = json.loads(raw_v1)
    v1_v2_rows.append({
        "note_id": target_id,
        "prompt": "V1",
        "root_cause": d_v1.get("root_cause"),
        "allowed": d_v1.get("root_cause") in ALLOWED_ROOT_CAUSES,
    })

    time.sleep(13)
    raw_v2 = analyse_note_raw(txt, prompt_text=build_craft_prompt(txt))
    d_v2 = json.loads(raw_v2)
    v1_v2_rows.append({
        "note_id": target_id,
        "prompt": "V2 (CRAFT)",
        "root_cause": d_v2.get("root_cause"),
        "allowed": d_v2.get("root_cause") in ALLOWED_ROOT_CAUSES,
    })

v1_v2_df = pd.DataFrame(v1_v2_rows)
print("Prompt comparison:\n", v1_v2_df)
print("\nDeployment choice: Deploy V2 (CRAFT) because it enforces the allowed label vocabulary and instructs the model to return 'other' rather than guessing on ambiguous notes.")

# (c) Total API calls
print(f"Total Gemini API calls used across Tasks 4.1-4.3: {api_call_count}")

### Task 4.4 — Merge with clean appointments and cross-tabulate

In [ ]:
appts_clean = pd.read_csv("appointments_clean.csv")
merged = notes_analysed.merge(appts_clean, on="appointment_id", how="left")
print("Merged rows:", len(merged))
print("\nCross-tabulation (root_cause vs reminder_sent):")
print(pd.crosstab(merged["root_cause"], merged["reminder_sent"]))